<a name='0'></a>
# How to Handle Categorical Data?

### Categorical Data Encoding

Real-world data comes in many different forms. Sometimes, while working with real-world datasets, we need to deal with **categorical data**, while in other cases we may not.

**Categorical data** is data whose feature values belong to a **limited number of categories**.

For example, consider a feature called `gender`. It can have two categories:

* `Male`
* `Female`

In many cases, categorical features contain **text values**. However, most machine learning algorithms work with **numerical inputs**. Therefore, we need to convert categorical values into numerical values before using them in machine learning models.

There are several techniques used to encode categorical features into numerical values:

1. **Mapping Method**
2. **Ordinal Encoding**
3. **Label Encoding**
4. **Pandas `get_dummies()`**
5. **One-Hot Encoding**

> **Note:** Some encoding techniques can produce the same or similar output. The main difference may be in their implementation and how they handle categories. The first three techniques generally produce numerical representations, while **One-Hot Encoding** produces a matrix containing `1`s and `0`s.

Let's implement each encoding technique one by one.


### Loading the Dataset

First, we import the required Python libraries.

* **Seaborn (`sns`)**: Used for data visualization and provides several built-in datasets.
* **Pandas (`pd`)**: Used for loading, manipulating, and analyzing tabular data.

These libraries are commonly used together for **data analysis and machine learning** tasks.


In [ ]:
# Loading the dataset

import seaborn as sns
import pandas as pd

### Loading the Titanic Dataset

The `sns.load_dataset('titanic')` function is used to load the **Titanic dataset** provided by Seaborn.

* `sns.load_dataset()` is used to load a built-in Seaborn dataset.
* `'titanic'` specifies the dataset we want to load.
* The loaded dataset is stored in a Pandas DataFrame named `titanic`.

The Titanic dataset contains information about passengers, such as **age, gender, passenger class, fare, and survival status**.


In [ ]:
titanic = sns.load_dataset('titanic')

### Viewing the Dataset

The `titanic.head()` function is used to display the **first 5 rows** of the Titanic dataset.

* `head()` is a Pandas function used to view the first few rows of a DataFrame.
* By default, it displays **5 rows**.
* It is useful for quickly checking the dataset's **columns, values, and overall structure**.


In [ ]:
titanic.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


### Checking Missing Values

The `titanic.isnull().sum()` function is used to **find the number of missing values** in each column of the Titanic dataset.

* `isnull()` checks each value and returns `True` if the value is missing and `False` otherwise.
* `sum()` counts the `True` values for each column.
* The result shows the **total number of missing values in each column**.

This is useful for identifying columns that contain missing data before performing data analysis or applying machine learning algorithms.


In [ ]:
titanic.isnull().sum()

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

### Understanding Dataset Information

The `titanic.info()` function is used to get a **summary of the Titanic dataset**.

It provides information such as:

* **Number of rows and columns**
* **Column names**
* **Number of non-null values** in each column
* **Data type** of each column
* **Memory usage** of the DataFrame

This function is useful for quickly understanding the **structure and data types** of a dataset before performing further analysis or preprocessing.


In [ ]:
titanic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB


You can see that even displaying information about the dataset, some features like `deck` or `class` have category as data type.

Let's peek at some categorical features in our data.

### Counting Categorical Values

The `titanic['sex'].value_counts()` function is used to **count the number of occurrences of each unique value** in the `sex` column.

* `titanic['sex']` selects the `sex` column from the dataset.
* `value_counts()` counts how many times each unique category appears.
* The result shows the number of **male and female** passengers.

This function is especially useful for understanding the **distribution of categorical data**.


In [ ]:
titanic['sex'].value_counts()

male      577
female    314
Name: sex, dtype: int64

In [ ]:
titanic['embarked'].value_counts()

S    644
C    168
Q     77
Name: embarked, dtype: int64

In [ ]:
titanic['class'].value_counts()

Third     491
First     216
Second    184
Name: class, dtype: int64

In [ ]:
titanic['who'].value_counts()

man      537
woman    271
child     83
Name: who, dtype: int64

In [ ]:
titanic['adult_male'].value_counts()

True     537
False    354
Name: adult_male, dtype: int64

In [ ]:
titanic['embark_town'].value_counts()

Southampton    644
Cherbourg      168
Queenstown      77
Name: embark_town, dtype: int64

In [ ]:
titanic['alone'].value_counts()

True     537
False    354
Name: alone, dtype: int64

In [ ]:
titanic['deck'].value_counts()

C    59
B    47
D    33
E    32
A    15
F    13
G     4
Name: deck, dtype: int64

## 1. Mapping Method

The **Mapping Method** is a simple and straightforward technique for converting categorical features into numerical values, especially when the feature contains a small number of categories.

For example, the `class` feature in the Titanic dataset contains three categories:

* `First`
* `Second`
* `Third`

To encode these categories, we create a **dictionary** where:

* **Keys** → Original categorical values
* **Values** → Numerical values assigned to those categories

We then use the `map()` function to replace the categorical values with their corresponding numerical values.

This method is useful when we want to manually define how each category should be represented numerically.

Here is how it is done:


In [ ]:
map_dict = {
    'First':0,
    'Second': 1,
    'Third': 2
}

In [ ]:
titanic['class'] = titanic['class'].map(map_dict)

In [ ]:
titanic.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,2,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,0,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,2,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,0,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,2,man,True,NaN,Southampton,no,True


In [ ]:
titanic['class'].value_counts()

2    491
0    216
1    184
Name: class, dtype: int64

As you can see, the class feature is encoded. Everywhere the class was `First`, it was replaced with 0. Samething happened to other classes.

<a name='2'></a>

## 2. Ordinal Encoding

**Ordinal Encoding** is a technique used to convert categorical data into **numerical values**.

It assigns a unique numerical value to each category. This method is especially useful when the categories have a **meaningful order or ranking**.

For example, the `class` feature contains:

* `First`
* `Second`
* `Third`

These categories have an order, so they can be represented using numerical values.

Let's implement it.


In [ ]:
from sklearn.preprocessing import OrdinalEncoder

cats_feats = titanic[['alive', 'alone']]

encoder = OrdinalEncoder()

cats_encoded = encoder.fit_transform(cats_feats)

The output of the encoder is a NumPy array. We can convert it back to the pandas dataframe.

In [ ]:
titanic[['alive', 'alone']] = pd.DataFrame(cats_encoded, columns=cats_feats.columns, index=cats_feats.index)
titanic.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,adult_male,deck,embark_town,alive,alone,man,woman
0,0,3,male,22.0,1,0,7.2500,S,2,True,NaN,Southampton,0.0,0.0,1,0
1,1,1,female,38.0,1,0,71.2833,C,0,False,C,Cherbourg,1.0,0.0,0,1
2,1,3,female,26.0,0,0,7.9250,S,2,False,NaN,Southampton,1.0,1.0,0,1
3,1,1,female,35.0,1,0,53.1000,S,0,False,C,Southampton,1.0,0.0,0,1
4,0,3,male,35.0,0,0,8.0500,S,2,True,NaN,Southampton,0.0,1.0,1,0


In [ ]:
encoder.categories_

[array(['no', 'yes'], dtype=object), array([False,  True])]

**Warning**: Ordinary Encoder can't handle missing values. It will be error. Try it on `embarked` and see...

<a name='3'></a>

## 3. Label Encoding

**Label Encoding** is a technique used to convert categorical values into **numerical values**.

According to the scikit-learn documentation, `LabelEncoder` is primarily intended for encoding **target variables**. However, it can also be used to convert categorical features into numerical values.

Label Encoding assigns a unique numerical label to each category.

One important point is that **LabelEncoder does not directly handle missing values**. Therefore, before applying Label Encoding, we need to handle or remove the missing values.

For simplicity, we will **drop the rows containing missing values** and then apply Label Encoding.

Let's implement it.


In [ ]:
titanic = sns.load_dataset('titanic')

titanic_cleaned = titanic.dropna()

In [ ]:
titanic_cleaned.isnull().sum()

survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
deck           0
embark_town    0
alive          0
alone          0
dtype: int64

In [ ]:
from sklearn.preprocessing import LabelEncoder

deck_feat = titanic_cleaned[['deck']]

label_encoder = LabelEncoder()

deck_encoded = label_encoder.fit_transform(deck_feat)

/Users/jean/opt/miniconda3/envs/tensor/lib/python3.7/site-packages/sklearn/utils/validation.py:63: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  return f(*args, **kwargs)


Same as ordinary encoder, the output of Label Encoder is a NumPy array.

In [ ]:
titanic_cleaned['deck'] = pd.DataFrame(deck_encoded, columns=deck_feat.columns, index=deck_feat.index)

titanic_cleaned.head()

/Users/jean/opt/miniconda3/envs/tensor/lib/python3.7/site-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """Entry point for launching an IPython kernel.


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,2,Cherbourg,yes,False
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,2,Southampton,yes,False
6,0,1,male,54.0,0,0,51.8625,S,First,man,True,4,Southampton,no,True
10,1,3,female,4.0,1,1,16.7000,S,Third,child,False,6,Southampton,yes,False
11,1,1,female,58.0,0,0,26.5500,S,First,woman,False,2,Southampton,yes,True


In [ ]:
label_encoder.classes_

array(['A', 'B', 'C', 'D', 'E', 'F', 'G'], dtype=object)

In [ ]:
titanic_cleaned['deck'].value_counts()

2    51
1    43
3    31
4    30
0    12
5    11
6     4
Name: deck, dtype: int64

<a name='4'></a>

## 4. Pandas Dummies

**Pandas Dummies** is a simple technique used to handle **categorical features** by converting each category into a separate feature (column).

For example, the `who` feature contains different categories. Using Pandas `get_dummies()` function, a new column is created for each category.

Each new column contains:

* `1` → If the row belongs to that category
* `0` → If the row does not belong to that category

This technique is useful for converting categorical data into a format that can be easily used by **machine learning algorithms**.

Let's apply it to the `who` feature.


In [ ]:
dummies = pd.get_dummies(titanic['who'], drop_first=True)

In [ ]:
titanic.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,2,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,0,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,2,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,0,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,2,man,True,NaN,Southampton,no,True


In [ ]:
titanic = pd.concat([titanic.drop('who',axis=1),dummies],axis=1)

In [ ]:
titanic.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,adult_male,deck,embark_town,alive,alone,man,woman
0,0,3,male,22.0,1,0,7.2500,S,2,True,NaN,Southampton,no,False,1,0
1,1,1,female,38.0,1,0,71.2833,C,0,False,C,Cherbourg,yes,False,0,1
2,1,3,female,26.0,0,0,7.9250,S,2,False,NaN,Southampton,yes,True,0,1
3,1,1,female,35.0,1,0,53.1000,S,0,False,C,Southampton,yes,False,0,1
4,0,3,male,35.0,0,0,8.0500,S,2,True,NaN,Southampton,no,True,1,0


In [ ]:
# Or you can do it at once with this code

#titanic[['man', 'woman']] = pd.get_dummies(titanic['who'], drop_first=True)

<a name='5'></a>
## 5. One Hot Encoding

This is the last encoding type of our list. It will convert a feature into one hot matrix. Additional features corresponding to the values of the given categories will be created. Basically same as dummies.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

one_hot = OneHotEncoder()

town_encoded = one_hot.fit_transform(titanic_cleaned[['embark_town']])

In [ ]:
one_hot.categories_

[array(['Cherbourg', 'Queenstown', 'Southampton'], dtype=object)]

In [ ]:
town_encoded

<182x3 sparse matrix of type '<class 'numpy.float64'>'
	with 182 stored elements in Compressed Sparse Row format>

The output of One hot encoder is a sparse matrix. We will need to convert it into NumPy array.

In [ ]:
town_encoded = town_encoded.toarray()

In [ ]:
columns = list(one_hot.categories_)

town_df = pd.DataFrame(town_encoded, columns =columns)

town_df.head()

,Cherbourg,Queenstown,Southampton
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,0.0,0.0,1.0
3,0.0,0.0,1.0
4,0.0,0.0,1.0


In [ ]:
len(town_df)

182

In [ ]:
len(titanic_cleaned)

182

In [ ]:
drop_embark = titanic_cleaned.drop('embark_town',axis=1)

drop_embark[['Cherbourg', 'Queenstown', 'Southampton']] = town_df

In [ ]:
drop_embark.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,alive,alone,Cherbourg,Queenstown,Southampton
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,2,yes,False,0.0,0.0,1.0
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,2,yes,False,0.0,0.0,1.0
6,0,1,male,54.0,0,0,51.8625,S,First,man,True,4,no,True,0.0,0.0,1.0
10,1,3,female,4.0,1,1,16.7000,S,Third,child,False,6,yes,False,0.0,0.0,1.0
11,1,1,female,58.0,0,0,26.5500,S,First,woman,False,2,yes,True,0.0,0.0,1.0


### Conclusion

These encoding techniques provide different ways to handle **categorical features** in real-world datasets.

Choosing the right encoding technique depends on the **type of categorical data** and the requirements of the machine learning algorithm.

Hopefully, these techniques will help you effectively handle different types of categorical features.
